In [ ]:
import re
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from typing import Iterable

In [ ]:
filename = "./article_links_16-09-25_12-21-53_with_contents"
articles_df = pd.read_csv(f"{filename}.csv")
articles_df

In [ ]:
author_pattern = r"By ([A-Za-z\s.]+).*"
social_media_pattern = r"@([A-Za-z]+)"

def get_page_datas(page_contents: Iterable[str]):
    soups = [BeautifulSoup(content, "html.parser") for content in page_contents]

    def get_titles():
        for soup in soups:
            title_element = soup.find("meta", property="og:title")
            yield title_element.get("content") if title_element else None
    
    def get_descriptions():
        for soup in soups:
            description_element = soup.find("meta", property="og:description")
            yield description_element.get("content") if description_element else None
    
    def get_published_times():
        for soup in soups:
            published_time_element = soup.find("meta", property="article:published_time")
            yield datetime.fromisoformat(published_time_element.get("content"))\
                if published_time_element\
                else None
    
    def get_modified_times():
        for soup in soups:
            modified_time_element = soup.find("meta", property="article:modified_time")
            yield datetime.fromisoformat(modified_time_element.get("content"))\
                if modified_time_element\
                else None

    def get_authors():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                author_match = re.search(author_pattern, credit)
                yield author_match.group(1).strip() if author_match else None
            else:
                yield None

    def get_social_medias():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                social_media_match = re.search(social_media_pattern, credit)
                yield social_media_match.group(1).strip() if social_media_match else None
            else:
                yield None
    
    def get_texts():
        for soup in soups:
            container = soup.find("div", "l-two-col__content")
            yield container.text.strip() if container else None
    
    def get_image_srcs_list():
        # Since encoding a list into a DataFrame can get weird,
        # I will save the image list as a single string, 
        # with each image src on a separate line.
        # Then when downloading images, I can decode this back into a list.
        for soup in soups:
            images_srcs: list[str] = [image.get("src") for image in soup.find_all("img")]
            images_srcs = [
                f"https://ufc.com{src}" if not src.startswith("https") else src
                for src in images_srcs 
            ]

            yield "\n".join(images_srcs)
    
    return {
        "title": get_titles(),
        "description": get_descriptions(),
        "published_time": get_published_times(),
        "modified_time": get_modified_times(),
        "author": get_authors(),
        "social_media": get_social_medias(),
        "text": get_texts(),
        "images_srcs": get_image_srcs_list(),
    }

In [ ]:
datas = get_page_datas(articles_df["content"])
for key in datas.keys():
    articles_df[key] = list(datas[key])

articles_df

In [ ]:
articles_df\
    .set_index("link")\
    .to_csv(f"{filename}_with_data.csv")